# Novelty Autoencoder — edge ONE STEP AHEAD + sparsity squeeze

**Settings:** T4 GPU, Internet On.

The design that escapes the leak-free impossibility:
- The edge **sees** b_t (one step ahead) -> it CAN carry the new content.
- A is **free** to the decoder + the edge is **squeezed** by a sparsity gate `alpha`.
  Copied tokens are cheap from (A + b_<t) -> gate closes; novel tokens need the edge
  -> gate opens. The open gates = 'what B adds', learned with NO labels.

**The thesis test** = ALPHA-LOCALIZATION: does `alpha` open on the inserted phrase
(NOVEL) far more than on COPIED tokens? Plus a same-space recovery eval.

In [ ]:
REPO_URL     = "https://github.com/haaaaaaayrewugrhyw/multihop-retrieval.git"
REPO_NAME    = "multihop-retrieval"
WORK_DIR     = f"/kaggle/working/{REPO_NAME}/complement_wiki"
DRIVE_FOLDER_ID = "1mMCf_lhYcw3CH_ttOWDLgSKZuPYqh5m5"
MAX_EXAMPLES = 120000
BETA         = 0.30           # sparsity squeeze; raise if gate doesn't localize
EDGE_DROPOUT = 0.5            # hide edge half the time -> forces the A+b_<t backbone
UPLOAD_TO_DRIVE = True

In [ ]:
# 1. GPU
import torch
if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Settings -> Accelerator -> T4 GPU")
if torch.cuda.get_device_properties(0).major < 7:
    raise RuntimeError("GPU is P100 - change to T4")
print(torch.cuda.get_device_name(0), "OK")

In [ ]:
# 2. Clone / pull + deps
import os
root = f"/kaggle/working/{REPO_NAME}"
os.system(f"git clone {REPO_URL} {root}" if not os.path.isdir(root) else f"cd {root} && git pull")
os.chdir(WORK_DIR)
os.system("pip install -q 'transformers>=4.35.0' 'datasets>=2.14.0' gdown")
print("cwd:", os.getcwd(), "| deps ready")

In [ ]:
# 3. Prefetch IteraTeR clean insertions (cached)
import sys, os
os.chdir(WORK_DIR); sys.path.insert(0, WORK_DIR)
sys.path.insert(0, f"{WORK_DIR}/../retrieval_v2")
import data_wikiedits as dw
trips = dw.load_triples(max_examples=MAX_EXAMPLES + 4000, cache=True)
print("clean-insertion triples:", len(trips))
for t in trips[:3]:
    print("  +", repr(t["inserted"]))

In [ ]:
# 4. Smoke test
import os
os.chdir(WORK_DIR)
os.system("python train_novelty.py --smoke")

In [ ]:
# 5. Full training (reports alpha-localization = the thesis test)
import os
os.chdir(WORK_DIR)
log = "/kaggle/working/novelty_train.log"
os.system(f"python train_novelty.py --max_examples {MAX_EXAMPLES} --beta {BETA} "
          f"--edge_dropout {EDGE_DROPOUT} 2>&1 | tee {log}")
print("\ndone")

In [ ]:
# 6. Same-space recovery eval (edge_novel vs enc_A floor vs enc_B ceiling)
import importlib.util, sys, os
os.chdir(WORK_DIR)
for m in ["eval_recovery_novelty", "novelty_ae", "data_wikiedits"]:
    sys.modules.pop(m, None)
spec = importlib.util.spec_from_file_location("eval_recovery_novelty", f"{WORK_DIR}/eval_recovery_novelty.py")
ern = importlib.util.module_from_spec(spec); sys.modules["eval_recovery_novelty"] = ern
sys.argv = ["eval_recovery_novelty.py", "--n", "1000", "--pool", str(MAX_EXAMPLES + 4000), "--n_distract", "19"]
spec.loader.exec_module(ern)
res = ern.main()

In [ ]:
# 7. Verify + optional rclone upload of checkpoint
import os, shutil
best = f"{WORK_DIR}/models/novelty_ae_best.pt"
if os.path.exists(best):
    print("checkpoint:", os.path.getsize(best)/1e6, "MB")
    shutil.copy(best, "/kaggle/working/novelty_ae_best.pt")
else:
    print("NOT FOUND - check log")
if UPLOAD_TO_DRIVE and os.path.exists(best):
    try:
        from kaggle_secrets import UserSecretsClient
        conf = UserSecretsClient().get_secret("RCLONE_CONF")
        os.makedirs("/root/.config/rclone", exist_ok=True)
        open("/root/.config/rclone/rclone.conf", "w").write(conf)
        os.system("curl -s https://downloads.rclone.org/rclone-current-linux-amd64.zip -o /tmp/r.zip")
        os.system("cd /tmp && unzip -qo r.zip && cp rclone-*-linux-amd64/rclone /usr/bin/ && chmod +x /usr/bin/rclone")
        rc = os.system(f"rclone copy {best} gdrive: --drive-root-folder-id {DRIVE_FOLDER_ID} -P")
        print("uploaded" if rc == 0 else f"rclone exit {rc}")
    except Exception as e:
        print("upload skipped:", e)

---
**How to read it:**
- `alpha NOVEL >> alpha COPIED` (cell 5 final report) = the squeeze made the gate open
  only on the added content -> **the thesis works**.
- `edge_novel >> enc_A` (cell 6) = the gated edge is a usable representation of the
  added content, in a comparable space.
- If gate doesn't localize: raise `BETA` (more squeeze) and retrain.